# 从零实现 EGNN 风格分子网络：E(n) 等变消息与属性预测

本 Notebook 用 PyTorch 基础算子手写径向消息、坐标更新、特征更新、分子图 batch、图级属性预测和有限梯度检查；不使用 PyG、DGL、e3nn、现成 GNN、注意力或 Transformer。

参考：[E(n) Equivariant Graph Neural Networks, ICML 2021](https://arxiv.org/abs/2102.09844)、[Tensor Field Networks, 2018](https://arxiv.org/abs/1802.08219) 与 [SE(3)-Transformer, NeurIPS 2020](https://arxiv.org/abs/2006.10503)。这里实现的是以标量特征和相对位移为核心的 EGNN 风格 E(n) 等变层，不是具有高阶不可约表示、球谐函数和张量积的完整 SE(3)-Transformer/e3nn 模型。受控分子任务只验证协议，不冒充量化化学泛化。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。
import copy, hashlib, json, math, random, warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED=6301  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。

def digest(payload): return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode()).hexdigest()  # 定义本节可复用的核心函数。
def tensor_semantic(t):  # 定义本节可复用的核心函数。
    v=t.detach().cpu().contiguous(); return {"dtype":str(v.dtype),"shape":list(v.shape),"sha256":hashlib.sha256(v.numpy().tobytes()).hexdigest()}  # 计算并保存当前步骤的中间状态。
def state_semantic(state): return {k:tensor_semantic(state[k]) for k in sorted(state)}  # 定义本节可复用的核心函数。

assert torch.get_num_threads()==1 and torch.device("cpu").type=="cpu"  # 用受控断言验证关键不变量。
assert tensor_semantic(torch.tensor([1.])) != tensor_semantic(torch.tensor([2.]))  # 用受控断言验证关键不变量。
assert not any(k in globals() for k in ("torch_geometric","dgl","e3nn"))  # 用受控断言验证关键不变量。


## 1. 分子图 batch 合同

`h:[N,F]` 是旋转不变的标量节点特征，`x:[N,3]` 是坐标，`edge_index:[2,E]` 表示 `src→dst` 消息，`batch:[N]` 标记分子。边不能跨分子；坐标与特征必须有限；每个分子至少一个节点；显式自环被拒绝，因为 EGNN 的径向方向在零位移处没有贡献且自环会重复中心信息。

零距离的不同节点是合法边界：使用平方距离，不做除法，所以应保持 finite。重复有向边被拒绝，避免无意重复计权。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class MolecularBatch:  # 定义承载本节状态与行为的数据结构。
    h: torch.Tensor  # 执行当前语句以推进本节示例。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    edge_index: torch.Tensor  # 执行当前语句以推进本节示例。
    batch: torch.Tensor  # 执行当前语句以推进本节示例。
    def validate(self):  # 定义本节可复用的核心函数。
        if self.h.ndim!=2 or self.x.shape!=(len(self.h),3) or not torch.isfinite(self.h).all() or not torch.isfinite(self.x).all():  # 按当前条件选择后续控制路径。
            raise ValueError("h/x 必须有限且 shape 正确")  # 遇到非法合同立即显式失败。
        if self.edge_index.ndim!=2 or self.edge_index.shape[0]!=2 or self.edge_index.dtype!=torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("edge_index 必须为 long [2,E]")  # 遇到非法合同立即显式失败。
        if self.batch.shape!=(len(self.h),) or self.batch.dtype!=torch.long or len(self.h)==0:  # 按当前条件选择后续控制路径。
            raise ValueError("batch 合同非法")  # 遇到非法合同立即显式失败。
        ids=self.batch.unique(sorted=True)  # 计算并保存当前步骤的中间状态。
        if not torch.equal(ids,torch.arange(len(ids))): raise ValueError("分子编号必须连续且非空")  # 按当前条件选择后续控制路径。
        if self.edge_index.numel():  # 按当前条件选择后续控制路径。
            src,dst=self.edge_index  # 计算并保存当前步骤的中间状态。
            if int(src.min())<0 or int(dst.min())<0 or int(src.max())>=len(self.h) or int(dst.max())>=len(self.h): raise ValueError("边越界")  # 按当前条件选择后续控制路径。
            if (src==dst).any(): raise ValueError("拒绝显式自环")  # 按当前条件选择后续控制路径。
            if not torch.equal(self.batch[src],self.batch[dst]): raise ValueError("检测到跨分子边")  # 按当前条件选择后续控制路径。
            pairs=list(zip(src.tolist(),dst.tolist()))  # 计算并保存当前步骤的中间状态。
            if len(pairs)!=len(set(pairs)): raise ValueError("重复有向边")  # 按当前条件选择后续控制路径。
        return self  # 返回当前分支计算出的结果。
    @property  # 为下方定义附加声明式配置。
    def num_graphs(self): return int(self.batch.max())+1  # 定义本节可复用的核心函数。
    def semantic(self):  # 定义本节可复用的核心函数。
        self.validate(); return {"h":tensor_semantic(self.h),"x":tensor_semantic(self.x),  # 执行当前语句以推进本节示例。
                                 "edge_index":tensor_semantic(self.edge_index),"batch":tensor_semantic(self.batch)}  # 执行当前语句以推进本节示例。

mol_probe=MolecularBatch(torch.eye(3),torch.tensor([[0.,0,0],[1.,0,0],[0.,1,0]]),  # 计算并保存当前步骤的中间状态。
                         torch.tensor([[0,1,1,2],[1,0,2,1]]),torch.zeros(3,dtype=torch.long)).validate()  # 计算并保存当前步骤的中间状态。
assert mol_probe.num_graphs==1  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    MolecularBatch(torch.ones(4,2),torch.zeros(4,3),torch.tensor([[0],[2]]),torch.tensor([0,0,1,1])).validate()  # 执行当前语句以推进本节示例。
    raise AssertionError("跨分子边未拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc: assert "跨分子" in str(exc)  # 捕获预期异常并验证失败分支。


## 2. EGNN 径向消息与坐标更新

对边 $j\to i$，平方距离 $r_{ij}^2=\|x_i-x_j\|^2$ 是任意平移、旋转和反射下的不变量：

$$m_{ij}=\phi_e(h_i,h_j,r_{ij}^2),\qquad
x_i'=x_i+\frac{1}{|\mathcal N(i)|}\sum_j(x_i-x_j)\phi_x(m_{ij}),$$
$$h_i'=h_i+\phi_h\left(h_i,\sum_jm_{ij}\right).$$

因为坐标只由相对向量乘标量更新，正交变换 $Q$（包括 `det(Q)=-1` 的反射）满足 $x'(Qx+t)=Qx'(x)+t$。这正是 E(n) 等变；它不编码手性敏感的赝标量，因此镜像异构体可能不可区分。


In [ ]:
class RadialMessage(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,hidden:int):  # 定义本节可复用的核心函数。
        super().__init__(); self.net=nn.Sequential(nn.Linear(2*hidden+1,hidden),nn.SiLU(),nn.Linear(hidden,hidden),nn.SiLU())  # 计算并保存当前步骤的中间状态。
    def forward(self,h_dst,h_src,r2):  # 定义本节可复用的核心函数。
        if r2.ndim!=2 or r2.shape[1]!=1: raise ValueError("r2 必须 [E,1]")  # 按当前条件选择后续控制路径。
        return self.net(torch.cat([h_dst,h_src,r2],dim=-1))  # 返回当前分支计算出的结果。

class EGNNLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,hidden:int):  # 定义本节可复用的核心函数。
        super().__init__(); self.message=RadialMessage(hidden)  # 计算并保存当前步骤的中间状态。
        self.coord=nn.Sequential(nn.Linear(hidden,hidden),nn.SiLU(),nn.Linear(hidden,1,bias=False))  # 计算并保存当前步骤的中间状态。
        self.feature=nn.Sequential(nn.Linear(2*hidden,hidden),nn.SiLU(),nn.Linear(hidden,hidden))  # 计算并保存当前步骤的中间状态。
    def forward(self,h,x,edge_index):  # 定义本节可复用的核心函数。
        if h.ndim!=2 or x.shape!=(len(h),3): raise ValueError("EGNN 输入 shape 非法")  # 按当前条件选择后续控制路径。
        src,dst=edge_index; relative=x[dst]-x[src]; r2=(relative.square()).sum(-1,keepdim=True)  # 计算并保存当前步骤的中间状态。
        m=self.message(h[dst],h[src],r2)  # 计算并保存当前步骤的中间状态。
        agg=torch.zeros_like(h); agg.index_add_(0,dst,m)  # 计算并保存当前步骤的中间状态。
        degree=torch.bincount(dst,minlength=len(h)).clamp_min(1).to(h.dtype).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
        delta=torch.zeros_like(x); delta.index_add_(0,dst,relative*self.coord(m)); delta=delta/degree  # 计算并保存当前步骤的中间状态。
        return h+self.feature(torch.cat([h,agg],-1)), x+delta  # 返回当前分支计算出的结果。

class EGNNEncoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,in_dim:int,hidden:int,depth:int):  # 定义本节可复用的核心函数。
        super().__init__(); self.embed=nn.Linear(in_dim,hidden); self.layers=nn.ModuleList([EGNNLayer(hidden) for _ in range(depth)])  # 计算并保存当前步骤的中间状态。
    def forward(self,mol:MolecularBatch):  # 定义本节可复用的核心函数。
        mol.validate(); h=F.silu(self.embed(mol.h)); x=mol.x  # 计算并保存当前步骤的中间状态。
        for layer in self.layers: h,x=layer(h,x,mol.edge_index)  # 遍历输入元素以累积或检查结果。
        return h,x  # 返回当前分支计算出的结果。

class MolecularPropertyModel(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,in_dim=3,hidden=24,depth=2):  # 定义本节可复用的核心函数。
        super().__init__(); self.encoder=EGNNEncoder(in_dim,hidden,depth); self.readout=nn.Sequential(nn.Linear(hidden,hidden),nn.SiLU(),nn.Linear(hidden,1))  # 计算并保存当前步骤的中间状态。
    def forward(self,mol:MolecularBatch):  # 定义本节可复用的核心函数。
        h,x=self.encoder(mol); pooled=torch.zeros(mol.num_graphs,h.shape[1]); pooled.index_add_(0,mol.batch,h)  # 计算并保存当前步骤的中间状态。
        pooled=pooled/torch.bincount(mol.batch,minlength=mol.num_graphs).to(h.dtype).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
        return self.readout(pooled).squeeze(1),x  # 返回当前分支计算出的结果。

model63=MolecularPropertyModel()  # 计算并保存当前步骤的中间状态。
y63,x63=model63(mol_probe)  # 计算并保存当前步骤的中间状态。
assert y63.shape==(1,) and x63.shape==(3,3) and torch.isfinite(x63).all()  # 用受控断言验证关键不变量。


## 3. 平移、旋转、反射 oracle

属性头只读取标量节点特征，因此输出应在 E(3) 下不变；内部坐标应等变。下面使用固定正交矩阵分别覆盖 `det=+1` 的旋转和 `det=-1` 的反射，并同时加入平移。测试比较实际数值，而不是只看维度。


In [ ]:
def transform_mol(mol,Q,t): return MolecularBatch(mol.h.clone(),mol.x@Q.T+t,mol.edge_index.clone(),mol.batch.clone()).validate()  # 定义本节可复用的核心函数。

theta=.7  # 计算并保存当前步骤的中间状态。
Qrot=torch.tensor([[math.cos(theta),-math.sin(theta),0.],[math.sin(theta),math.cos(theta),0.],[0.,0.,1.]])  # 计算并保存当前步骤的中间状态。
Qref=torch.diag(torch.tensor([-1.,1.,1.]))  # 计算并保存当前步骤的中间状态。
t=torch.tensor([3.,-2.,.5])  # 计算并保存当前步骤的中间状态。
model63.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    y0,x0=model63(mol_probe)  # 计算并保存当前步骤的中间状态。
    yr,xr=model63(transform_mol(mol_probe,Qrot,t))  # 计算并保存当前步骤的中间状态。
    yf,xf=model63(transform_mol(mol_probe,Qref,t))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(y0,yr,atol=2e-6) and torch.allclose(y0,yf,atol=2e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(xr,x0@Qrot.T+t,atol=2e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(xf,x0@Qref.T+t,atol=2e-6)  # 用受控断言验证关键不变量。
assert torch.det(Qrot)>0 and torch.det(Qref)<0  # 用受控断言验证关键不变量。


## 4. 节点置换、零距离与中心性质

节点重排后，属性不变、节点坐标按相同置换等变。零距离边的 `relative=0`，坐标增量自然为零且没有除零。由于更新只使用内部相对位移，每个分子的坐标质心在成对对称边、相同标量权重时近似守恒；一般有向/非对称邻接并不保证质心严格不变，因此生产前必须明确边对称合同。


In [ ]:
def permute_mol(mol,perm):  # 定义本节可复用的核心函数。
    if sorted(perm.tolist())!=list(range(len(mol.h))): raise ValueError("非法置换")  # 按当前条件选择后续控制路径。
    inv=torch.empty_like(perm); inv[perm]=torch.arange(len(perm))  # 计算并保存当前步骤的中间状态。
    return MolecularBatch(mol.h[perm],mol.x[perm],inv[mol.edge_index],mol.batch[perm]).validate()  # 返回当前分支计算出的结果。

perm=torch.tensor([2,0,1])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    yp,xp=model63(permute_mol(mol_probe,perm))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(yp,y0,atol=2e-6) and torch.allclose(xp,x0[perm],atol=2e-6)  # 用受控断言验证关键不变量。

zero_mol=MolecularBatch(torch.tensor([[1.,0,0],[0.,1,0]]),torch.zeros(2,3),  # 计算并保存当前步骤的中间状态。
                        torch.tensor([[0,1],[1,0]]),torch.zeros(2,dtype=torch.long)).validate()  # 计算并保存当前步骤的中间状态。
with torch.no_grad(): yz,xz=model63(zero_mol)  # 在受管理的上下文中执行操作。
assert torch.isfinite(yz).all() and torch.isfinite(xz).all()  # 用受控断言验证关键不变量。
assert torch.equal(xz,torch.zeros_like(xz))  # 用受控断言验证关键不变量。
assert float((x0-mol_probe.x).abs().max()) > 0  # 非零距离边确实走过坐标更新路径


## 5. 标量能量梯度与“力”检查

若模型输出标量能量 $E(x)$，自动微分力为 $F_i=-\partial E/\partial x_i$。平移不变性意味着理想情况下总力接近 0。这里检查梯度存在、有限且总和近零；这只是数值/对称性 oracle，不代表模型输出具备真实物理能量单位或保守力场精度。


In [ ]:
grad_x=mol_probe.x.clone().requires_grad_(True)  # 计算并保存当前步骤的中间状态。
grad_mol=MolecularBatch(mol_probe.h,grad_x,mol_probe.edge_index,mol_probe.batch)  # 计算并保存当前步骤的中间状态。
energy,_=model63(grad_mol); force=-torch.autograd.grad(energy.sum(),grad_x,create_graph=False)[0]  # 计算并保存当前步骤的中间状态。
assert force.shape==(3,3) and torch.isfinite(force).all()  # 用受控断言验证关键不变量。
assert torch.allclose(force.sum(0),torch.zeros(3),atol=2e-5)  # 用受控断言验证关键不变量。
assert grad_x.grad is None  # autograd.grad 不累积到 .grad


## 6. 非周期分子 fixture、显式 group split 与受控属性任务

生成 20 个三至五节点的小分子，全连接双向边。原子类型模式可以重复，但每个 `index` 使用独立 generator 对基础构型加入小幅连续扰动，且半径含非周期 index 项；因此不同 original group 不会每 12 个样本精确复现。标签主要由原子类型比例决定，并加入很弱的几何项。

split 的单位是完整 `original_molecule_id`，不是节点或旋转 view。代码同时断言 group ID 不交叉、完整 tensor 摘要不交叉。随后用 packed 两分子干预检查：只修改第一张图的特征，第二张图的标量输出和坐标必须逐位不变，防止聚合时漏用 `batch` 或出现跨图边。

这个任务刻意简单，使 fresh-kernel 能快速验证反向传播与不变量；它不是 QM9、OC20 或药物发现 benchmark。


In [ ]:
def make_molecule(index:int):  # 定义本节可复用的核心函数。
    if not isinstance(index,int) or index<0: raise ValueError("molecule index 非法")  # 按当前条件选择后续控制路径。
    n=3+index%3; types=torch.tensor([(index+j)%3 for j in range(n)])  # 计算并保存当前步骤的中间状态。
    h=F.one_hot(types,3).float()  # 计算并保存当前步骤的中间状态。
    generator=torch.Generator().manual_seed(SEED+2000+index)  # 计算并保存当前步骤的中间状态。
    angle=torch.linspace(0,2*math.pi*(n-1)/n,n)  # 计算并保存当前步骤的中间状态。
    radius=.68+.0037*index  # 计算并保存当前步骤的中间状态。
    base=torch.stack([radius*torch.cos(angle),radius*torch.sin(angle),.055*torch.arange(n)],1)  # 计算并保存当前步骤的中间状态。
    x=base+.012*torch.randn(n,3,generator=generator)  # 计算并保存当前步骤的中间状态。
    edges=[(u,v) for u in range(n) for v in range(n) if u!=v]  # 计算并保存当前步骤的中间状态。
    edge=torch.tensor(edges,dtype=torch.long).T  # 计算并保存当前步骤的中间状态。
    r2=(x[edge[0]]-x[edge[1]]).square().sum(1).mean()  # 计算并保存当前步骤的中间状态。
    target=1.5*h[:,0].mean()+.5*h[:,1].mean()-.5*h[:,2].mean()+.03*r2  # 计算并保存当前步骤的中间状态。
    return MolecularBatch(h,x,edge,torch.zeros(n,dtype=torch.long)).validate(),target  # 返回当前分支计算出的结果。

def pack_molecules(items):  # 定义本节可复用的核心函数。
    if not items: raise ValueError("不能打包空分子列表")  # 按当前条件选择后续控制路径。
    hs=[]; xs=[]; es=[]; bs=[]; off=0  # 计算并保存当前步骤的中间状态。
    for gid,m in enumerate(items):  # 遍历输入元素以累积或检查结果。
        m.validate(); hs.append(m.h);xs.append(m.x);es.append(m.edge_index+off);bs.append(torch.full((len(m.h),),gid,dtype=torch.long));off+=len(m.h)  # 计算并保存当前步骤的中间状态。
    return MolecularBatch(torch.cat(hs),torch.cat(xs),torch.cat(es,1),torch.cat(bs)).validate()  # 返回当前分支计算出的结果。

dataset=[make_molecule(i) for i in range(20)]  # 计算并保存当前步骤的中间状态。
group_ids63=[f"original-molecule-{i:03d}" for i in range(20)]  # 计算并保存当前步骤的中间状态。
train_indices63=list(range(15)); test_indices63=list(range(15,20))  # 计算并保存当前步骤的中间状态。
train_groups63={group_ids63[i] for i in train_indices63}; test_groups63={group_ids63[i] for i in test_indices63}  # 计算并保存当前步骤的中间状态。
train_semantics63={digest(dataset[i][0].semantic()) for i in train_indices63}  # 计算并保存当前步骤的中间状态。
test_semantics63={digest(dataset[i][0].semantic()) for i in test_indices63}  # 计算并保存当前步骤的中间状态。
assert train_groups63.isdisjoint(test_groups63)  # 用受控断言验证关键不变量。
assert len(train_semantics63)==len(train_indices63) and len(test_semantics63)==len(test_indices63)  # 用受控断言验证关键不变量。
assert train_semantics63.isdisjoint(test_semantics63)  # 用受控断言验证关键不变量。

train_mol=pack_molecules([dataset[i][0] for i in train_indices63]); test_mol=pack_molecules([dataset[i][0] for i in test_indices63])  # 计算并保存当前步骤的中间状态。
ytrain=torch.stack([dataset[i][1] for i in train_indices63]); ytest=torch.stack([dataset[i][1] for i in test_indices63])  # 计算并保存当前步骤的中间状态。
model63=MolecularPropertyModel(); opt=torch.optim.Adam(model63.parameters(),lr=.015); history=[]  # 计算并保存当前步骤的中间状态。
for _ in range(40):  # 遍历输入元素以累积或检查结果。
    pred,_=model63(train_mol); loss=F.mse_loss(pred,ytrain); opt.zero_grad();loss.backward();opt.step();history.append(float(loss))  # 计算并保存当前步骤的中间状态。
with torch.no_grad(): test_pred,_=model63(test_mol); mae63=float((test_pred-ytest).abs().mean())  # 在受管理的上下文中执行操作。
assert history[-1] < history[0]*.05 and mae63 < .12  # 用受控断言验证关键不变量。
assert torch.isfinite(test_pred).all()  # 用受控断言验证关键不变量。

pair63=pack_molecules([dataset[0][0],dataset[1][0]])  # 计算并保存当前步骤的中间状态。
changed_h63=pair63.h.clone(); changed_h63[pair63.batch==0,0]+=.5  # 计算并保存当前步骤的中间状态。
changed_pair63=MolecularBatch(changed_h63,pair63.x.clone(),pair63.edge_index.clone(),pair63.batch.clone()).validate()  # 计算并保存当前步骤的中间状态。
with torch.no_grad(): pair_y63,pair_x63=model63(pair63); changed_y63,changed_x63=model63(changed_pair63)  # 在受管理的上下文中执行操作。
second63=pair63.batch==1  # 计算并保存当前步骤的中间状态。
assert torch.allclose(pair_y63[1],changed_y63[1],atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(pair_x63[second63],changed_x63[second63],atol=1e-7)  # 用受控断言验证关键不变量。
assert not torch.allclose(pair_y63[0],changed_y63[0])  # 用受控断言验证关键不变量。
print({"mse_first":round(history[0],5),"mse_last":round(history[-1],5),"controlled_test_mae":round(mae63,5)})  # 执行当前语句以推进本节示例。


## 7. 复杂度、推理与完整 SE(3) 模型的差异

每层对 $E$ 条边计算 MLP，约为 $O(EH^2+NH^2)$ 时间、$O(EH+NH)$ 激活内存；全连接分子图有 $E=O(N^2)$，大体系必须使用半径图、cell list 与 neighbor cap。推理时属性只返回图级标量；若需要优化构象，还要保存更新坐标或对能量求梯度，并验证单位、边界条件与数值稳定性。

EGNN 的标量通道和相对向量更新天然覆盖平移、旋转、反射，但不显式维护 $l>0$ 的不可约表示，也缺少球谐/张量积、角动量耦合和手性敏感通道。对方向性轨道、张量属性、手性和高精度力场，应评估 SE(3)-Transformer、Tensor Field Network、NequIP/MACE 等更完整模型，而不是把本实现等同于它们。


## 8. 发布合同：坐标 recipe、图语义与带外锚

manifest 绑定输入 schema、坐标单位/中心策略/邻接 recipe、训练/test 分子张量、split、模型配置以及 state key/dtype/shape/bytes。这里坐标单位是“合成无量纲”，不做输入中心化；模型依赖相对位置所以仍平移不变。

`PublishedMolecularModel` 从实际 `MolecularBatch` 重算完整语义，只接受注册 train/test fixture。真实服务一般不能绑定每个输入值，但至少应重算并验证 schema、单位、原子词表、邻接/截断/周期边界 recipe；这里绑定值是为了演示最严格的可复现实验制品。


In [ ]:
CONFIG63={"in_dim":3,"hidden":24,"depth":2}  # 计算并保存当前步骤的中间状态。
COORD63={"dimension":3,"unit":"synthetic-unitless","centering":"none","edges":"complete-directed-no-self",  # 计算并保存当前步骤的中间状态。
         "distance":"squared","coordinate_update":"mean-incoming-relative-times-scalar"}  # 执行当前语句以推进本节示例。
SPLIT63={"train_indices":train_indices63,"test_indices":test_indices63,"unit":"whole_original_molecule","train_groups":sorted(train_groups63),"test_groups":sorted(test_groups63),"semantic_overlap":0}  # 计算并保存当前步骤的中间状态。
state63={k:v.detach().cpu().clone() for k,v in model63.state_dict().items()}  # 计算并保存当前步骤的中间状态。
manifest63={"schema":"h[N,3]-x[N,3]-edge[2,E]-batch[N]/v1","coordinate_recipe":COORD63,"split":SPLIT63,  # 计算并保存当前步骤的中间状态。
            "train_graph":digest(train_mol.semantic()),"test_graph":digest(test_mol.semantic()),  # 执行当前语句以推进本节示例。
            "config":CONFIG63,"state":state_semantic(state63)}  # 执行当前语句以推进本节示例。
artifact63={"release_id":"egnn-63-v1","manifest":manifest63,"state":state63}  # 计算并保存当前步骤的中间状态。
def artifact_digest63(a): return digest({"release_id":a["release_id"],"manifest":a["manifest"]})  # 定义本节可复用的核心函数。
_TRUSTED_RELEASES63=MappingProxyType({"egnn-63-v1":artifact_digest63(artifact63)})  # 计算并保存当前步骤的中间状态。

class PublishedMolecularModel(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,model,allowed): super().__init__();self.model=model.eval();self.allowed=allowed  # 定义本节可复用的核心函数。
    def forward(self,mol:MolecularBatch):  # 定义本节可复用的核心函数。
        actual=digest(mol.validate().semantic())  # 计算并保存当前步骤的中间状态。
        if actual not in self.allowed: raise ValueError("输入分子语义未注册")  # 按当前条件选择后续控制路径。
        with torch.no_grad(): return self.model(mol)[0]  # 在受管理的上下文中执行操作。

def load63(a):  # 定义本节可复用的核心函数。
    rid=a.get("release_id")  # 计算并保存当前步骤的中间状态。
    if rid not in _TRUSTED_RELEASES63 or artifact_digest63(a)!=_TRUSTED_RELEASES63[rid]: raise ValueError("带外 registry 拒绝 release")  # 按当前条件选择后续控制路径。
    if a["manifest"]["state"]!=state_semantic(a["state"]): raise ValueError("state 描述不匹配")  # 按当前条件选择后续控制路径。
    if a["manifest"]["coordinate_recipe"]!=COORD63: raise ValueError("坐标 recipe 不匹配")  # 按当前条件选择后续控制路径。
    m=MolecularPropertyModel(**a["manifest"]["config"]);m.load_state_dict(a["state"],strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedMolecularModel(m,(a["manifest"]["train_graph"],a["manifest"]["test_graph"]))  # 返回当前分支计算出的结果。

published63=load63(artifact63)  # 计算并保存当前步骤的中间状态。
assert published63(test_mol).shape==(5,)  # 用受控断言验证关键不变量。
attack63=copy.deepcopy(artifact63);attack63["manifest"]["coordinate_recipe"]["unit"]="angstrom"  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load63(attack63);raise AssertionError("整体重签未拒绝")  # 执行当前语句以推进本节示例。
except ValueError as exc: assert "registry" in str(exc)  # 捕获预期异常并验证失败分支。
try:  # 尝试执行可能失败的受控操作。
    published63(mol_probe);raise AssertionError("未注册输入未拒绝")  # 执行当前语句以推进本节示例。
except ValueError as exc: assert "语义" in str(exc)  # 捕获预期异常并验证失败分支。


## 9. 生产检查清单与结论边界

必须补齐：真实原子/键特征词表；Å/eV 等单位合同；半径图与周期边界；双向边一致性；力/能量联合损失；旋转/reflection/置换属性测试；手性任务；长程相互作用；数据按分子 scaffold/时间切分；多 seed 与真实 benchmark；混合精度/大坐标溢出；邻居缓存；制品签名、数据许可和回滚。

本 Notebook 证明的是手写 EGNN 风格层在数值上满足平移、旋转、反射、置换与有限梯度 oracle，并能完成受控任务；不证明化学精度，也不声称与完整 SE(3) 等变网络等价。


In [ ]:
assert set(manifest63)=={"schema","coordinate_recipe","split","train_graph","test_graph","config","state"}  # 用受控断言验证关键不变量。
assert mae63 < .12 and torch.isfinite(force).all()  # 用受控断言验证关键不变量。
assert len(_TRUSTED_RELEASES63)==1  # 用受控断言验证关键不变量。
print("EGNN 63：等变性、梯度、训练与发布 oracle 通过。")  # 执行当前语句以推进本节示例。
